In [11]:
!pip install pandas numpy scikit-learn matplotlib seaborn vaderSentiment textblob fuzzywuzzy[speedup]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.9/159.9 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 98.2 MB/s eta 0:00:00


In [5]:
from google.colab import files
uploaded = files.upload()

Saving Sentiment analysis.csv to Sentiment analysis.csv
Saving Player_Stats.csv to Player_Stats.csv
Saving Market value data.csv to Market value data.csv
Saving Injury data.csv to Injury data.csv


In [7]:

import pandas as pd
import numpy as np

player_stats = pd.read_csv('Player_Stats.csv', encoding='latin1', sep=';')
player_stats.columns = player_stats.columns.str.strip()

player_stats.drop_duplicates(inplace=True)

num_cols = player_stats.select_dtypes(include=[np.number]).columns
player_stats[num_cols] = player_stats[num_cols].fillna(player_stats[num_cols].mean())

cat_cols = player_stats.select_dtypes(include=['object']).columns
player_stats[cat_cols] = player_stats[cat_cols].fillna(player_stats[cat_cols].mode().iloc[0])

player_stats = player_stats.assign(
    Performance_Index=(
        (player_stats["Goals"].astype(float) + player_stats["Assists"].astype(float)) /
        player_stats["90s"].replace(0, np.nan)
    ),
    Shot_Accuracy=(
        player_stats["SoT"].astype(float) / player_stats["Shots"].replace(0, np.nan)
    ),
    Pass_Efficiency=(
        player_stats["PasTotCmp"].astype(float) / player_stats["PasTotAtt"].replace(0, np.nan)
    )
).copy()

print("Player Stats cleaned & engineered:")
print(player_stats.head())

Player Stats cleaned & engineered:
   Rk             Player Nation   Pos         Squad            Comp  Age  \
0   1   Brenden Aaronson    USA  MFFW  Leeds United  Premier League   22   
1   2   Yunis Abdelhamid    MAR    DF         Reims         Ligue 1   35   
2   3      Himad Abdelli    FRA  MFFW        Angers         Ligue 1   23   
3   4  Salis Abdul Samed    GHA    MF          Lens         Ligue 1   22   
4   5    Laurent Abergel    FRA    MF       Lorient         Ligue 1   30   

   Born  MP  Starts  ...  PKwon  PKcon    OG  Recov  AerWon  AerLost  AerWon%  \
0  2000  20      19  ...    0.0    0.0  0.00   4.86    0.34     1.19     22.2   
1  1987  22      22  ...    0.0    0.0  0.00   6.64    2.18     1.23     64.0   
2  1999  14       8  ...    0.0    0.0  0.00   8.14    0.93     1.05     47.1   
3  2000  20      20  ...    0.0    0.0  0.05   6.60    0.50     0.50     50.0   
4  1993  15      15  ...    0.0    0.0  0.00   6.51    0.31     0.39     44.4   

   Performance_Index 

In [8]:
import pandas as pd
import numpy as np

injury_data = pd.read_csv('Injury data.csv')
injury_data.drop_duplicates(inplace=True)

num_cols = injury_data.select_dtypes(include=[np.number]).columns
injury_data[num_cols] = injury_data[num_cols].fillna(injury_data[num_cols].mean())

injury_data["Injury_Risk_Score"] = np.log1p(injury_data["season_days_injured"])

injury_data["Games_Missed_Ratio"] = (
    1 - (injury_data["season_games_played"] / injury_data["season_matches_in_squad"].replace(0, np.nan))
)

injury_data["Injury_Severity_Index"] = (
    injury_data["total_days_injured"] / injury_data["total_games_played"].replace(0, np.nan)
)

print("Injury Data cleaned & engineered:")
print(injury_data.head())

Injury Data cleaned & engineered:
            p_id2  start_year  season_days_injured  total_days_injured  \
0   aaronconnolly        2019                   13                 161   
1   aaronconnolly        2020                   71                 161   
2  aaroncresswell        2016                   95                 226   
3  aaroncresswell        2018                   87                 226   
4  aaroncresswell        2019                   35                 226   

   season_minutes_played  season_games_played  season_matches_in_squad  \
0                 1312.0                   24                       28   
1                  836.0                   17                       28   
2                 2247.0                   26                       27   
3                 1680.0                   20                       27   
4                 2870.0                   31                       31   

   total_minutes_played  total_games_played         dob  ...  \
0           

In [12]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer


sentiment_data = pd.read_csv('Sentiment analysis.csv')
# Keeps only columns not starting with 'Unnamed'.
sentiment_data = sentiment_data.loc[:, ~sentiment_data.columns.str.contains('^Unnamed')]

sentiment_data.fillna({"Text": "", "Sentiment": "Neutral"}, inplace=True)

# Returns a dictionary of sentiment scores
analyzer = SentimentIntensityAnalyzer()
sentiment_data["Sentiment_Score"] = sentiment_data["Text"].astype(str).apply(
    lambda x: analyzer.polarity_scores(x)["compound"]
)


sentiment_map = {"Positive": 1, "Neutral": 0, "Negative": -1}
sentiment_data["Sentiment_Label"] = sentiment_data["Sentiment"].map(sentiment_map)

# group text by user and computes avg. sentiment score & sentiment volume
player_sentiment = sentiment_data.groupby("User").agg(
    Avg_Sentiment_Score=("Sentiment_Score", "mean"),
    Sentiment_Volume=("Text", "count")
).reset_index()

print("Sentiment Data cleaned & engineered:")
print(player_sentiment.head())

Sentiment Data cleaned & engineered:
                      User  Avg_Sentiment_Score  Sentiment_Volume
0         AbyssOfTime                    0.5106                 1
1         AcceptanceSeeker               0.5106                 1
2        AdeleConcertGoer                0.3612                 1
3   AdeleMelodyTearjerker                0.2500                 1
4         AdventureAwaits                0.6705                 1


In [15]:
import pandas as pd
import numpy as np

market_value = pd.read_csv('Market value data.csv')

# Check the column names after loading
print("Columns in market_value DataFrame:", market_value.columns)

market_value.rename(columns={"Name": "Player", "market_value": "Market_Value"}, inplace=True)
market_value["Player"] = market_value["Player"].str.replace("Ã©", "é").str.replace("Ã±", "ñ")
#  converting market value column to numeric type and invalid values are converted into NaN
market_value["Market_Value"] = pd.to_numeric(market_value["Market_Value"], errors="coerce")

market_value.dropna(subset=["Player", "Market_Value"], inplace=True)

# using log1p() for normalizing the distribution
market_value["Market_Value_Log"] = np.log1p(market_value["Market_Value"])

print("Market Value cleaned & engineered:")
print(market_value.head())

Columns in market_value DataFrame: Index(['Club', 'Name', 'market_value'], dtype='object')
Market Value cleaned & engineered:
          Club               Player  Market_Value  Market_Value_Log
0     Man City       Erling Haaland          60.0          4.110874
1      Chelsea       Enzo Fernández         121.0          4.804021
2      Man Utd               Antony          95.0          4.564348
3      Chelsea        Wesley Fofana          80.4          4.399375
4  Real Madrid  Aurélien Tchouaméni          80.0          4.394449
